## Data Pipeline Step 1: Multi-Source CSV Ingestion, Schema Validation, and Consolidated Dataset Generation (WFO)

In [ ]:
import pandas as pd
import os

# ==============================
# CONFIG
# ==============================
FOLDER_PATH = r"C:\Users\Ranja\OneDrive\Documents\WFO_JOB_PROJECT\data"
OUTPUT_FILE = "merged_wfo_data.csv"

# ==============================
# STEP 1: GET FILES
# ==============================
files = [
    f for f in os.listdir(FOLDER_PATH)
    if f.endswith(".csv") and f.startswith("WFO")
]

files = sorted(files)

if len(files) == 0:
    raise Exception("No CSV files found")

print(f"Total files detected: {len(files)}")

# ==============================
# STEP 2: SCHEMA VALIDATION
# ==============================
base_columns = None
file_row_counts = {}

for file in files:
    path = os.path.join(FOLDER_PATH, file)
    temp_df = pd.read_csv(path)

    file_row_counts[file] = len(temp_df)

    if base_columns is None:
        base_columns = list(temp_df.columns)
    else:
        if list(temp_df.columns) != base_columns:
            raise Exception(f"Column mismatch detected in file: {file}")

print("Schema validation passed")

# ==============================
# STEP 3: MERGE (STRICT APPEND)
# ==============================
df_list = []

for file in files:
    path = os.path.join(FOLDER_PATH, file)
    temp_df = pd.read_csv(path)

    df_list.append(temp_df)

merged_df = pd.concat(df_list, ignore_index=True)

# ==============================
# STEP 4: VALIDATION
# ==============================
total_input_rows = sum(file_row_counts.values())
total_merged_rows = len(merged_df)

print("\n--- FILE LEVEL COUNTS ---")
for file, count in file_row_counts.items():
    print(f"{file}: {count} rows")

print("\n--- SUMMARY ---")
print(f"Total input rows: {total_input_rows}")
print(f"Total merged rows: {total_merged_rows}")

if total_input_rows != total_merged_rows:
    raise Exception("Row count mismatch after merge")

print("Row count validation passed")

# ==============================
# STEP 5: SAVE OUTPUT
# ==============================
output_path = os.path.join(FOLDER_PATH, OUTPUT_FILE)

merged_df.to_csv(output_path, index=False)

print(f"Merged file saved at: {output_path}")

Total files detected: 10
Schema validation passed

--- FILE LEVEL COUNTS ---
WFO.csv: 1000 rows
WFO_P10.csv: 1000 rows
WFO_P2.csv: 1000 rows
WFO_P3.csv: 1000 rows
WFO_P4.csv: 1000 rows
WFO_P5.csv: 1000 rows
WFO_P6.csv: 1000 rows
WFO_P7.csv: 1000 rows
WFO_P8.csv: 1000 rows
WFO_P9.csv: 1000 rows

--- SUMMARY ---
Total input rows: 10000
Total merged rows: 10000
Row count validation passed
Merged file saved at: C:\Users\Ranja\OneDrive\Documents\WFO_JOB_PROJECT\data\merged_wfo_data.csv


## Step 2: Fix Weekday, Week Number, Month, and Year Using Date

In [2]:
import pandas as pd
import os

# ==============================
# CONFIG
# ==============================
BASE_PATH = r"C:\Users\Ranja\OneDrive\Documents\WFO_JOB_PROJECT\data"

INPUT_FILE = os.path.join(BASE_PATH, "merged_wfo_data.csv")
OUTPUT_FILE = os.path.join(BASE_PATH, "wfo_data.csv")

# ==============================
# STEP 1: LOAD DATA
# ==============================
df = pd.read_csv(INPUT_FILE)

print("File loaded successfully")
print("Total rows:", len(df))
print("Total columns:", len(df.columns))

# Store initial row count
initial_row_count = len(df)

# ==============================
# STEP 2: VALIDATE REQUIRED COLUMNS
# ==============================
required_cols = ['Date', 'Weekday', 'Week_Number', 'Month', 'Year']

missing_cols = [col for col in required_cols if col not in df.columns]

if len(missing_cols) > 0:
    raise Exception(f"Missing columns: {missing_cols}")

print("Required columns validation passed")

# ==============================
# STEP 3: CONVERT DATE COLUMN
# ==============================
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

invalid_dates = df['Date'].isna().sum()

if invalid_dates > 0:
    raise Exception(f"Invalid Date values found: {invalid_dates}")

print("Date column conversion successful")

# ==============================
# STEP 4: CAPTURE OLD VALUES (FOR VALIDATION)
# ==============================
old_values = df[['Weekday', 'Week_Number', 'Month', 'Year']].copy()

# ==============================
# STEP 5: OVERWRITE INCORRECT VALUES
# ==============================
df['Weekday'] = df['Date'].dt.day_name()

df['Week_Number'] = df['Date'].dt.isocalendar().week.astype(int)

df['Month'] = df['Date'].dt.month_name()

df['Year'] = df['Date'].dt.year

print("Date-derived columns corrected")

# ==============================
# STEP 6: VALIDATION
# ==============================

# Null check
null_check = df[['Weekday', 'Week_Number', 'Month', 'Year']].isnull().sum()

print("Null check after transformation:")
print(null_check)

if null_check.sum() > 0:
    raise Exception("Null values found after date correction")

# Row count validation
final_row_count = len(df)

print("Initial row count:", initial_row_count)
print("Final row count:", final_row_count)

if initial_row_count != final_row_count:
    raise Exception("Row count mismatch after transformation")

# ==============================
# STEP 7: CHANGE VALIDATION
# ==============================

comparison = (
    (old_values['Weekday'] != df['Weekday']) |
    (old_values['Week_Number'] != df['Week_Number']) |
    (old_values['Month'] != df['Month']) |
    (old_values['Year'] != df['Year'])
)

changed_rows = comparison.sum()

print("Rows where values changed:", changed_rows)

if changed_rows == 0:
    print("Warning: No changes detected. Check data logic.")
else:
    print("Change validation passed")

# ==============================
# STEP 8: SAMPLE VERIFICATION
# ==============================
print("Sample records:")
print(df[['Date', 'Weekday', 'Week_Number', 'Month', 'Year']].head())

# ==============================
# STEP 9: SAVE FINAL OUTPUT
# ==============================
df.to_csv(OUTPUT_FILE, index=False)

print("Final file saved at:")
print(OUTPUT_FILE)

File loaded successfully
Total rows: 10000
Total columns: 30
Required columns validation passed
Date column conversion successful
Date-derived columns corrected
Null check after transformation:
Weekday        0
Week_Number    0
Month          0
Year           0
dtype: int64
Initial row count: 10000
Final row count: 10000
Rows where values changed: 10000
Change validation passed
Sample records:
        Date    Weekday  Week_Number      Month  Year
0 2023-09-29     Friday           39  September  2023
1 2022-10-27   Thursday           43    October  2022
2 2022-10-16     Sunday           41    October  2022
3 2021-06-23  Wednesday           25       June  2021
4 2021-10-24     Sunday           42    October  2021


C:\Users\Ranja\AppData\Local\Temp\ipykernel_3620\3085048957.py:39: UserWarning: Parsing dates in DD/MM/YYYY format when dayfirst=False (the default) was specified. This may lead to inconsistently parsed dates! Specify a format to ensure consistent parsing.
  df['Date'] = pd.to_datetime(df['Date'], errors='coerce')


Final file saved at:
C:\Users\Ranja\OneDrive\Documents\WFO_JOB_PROJECT\data\wfo_data.csv


## Step 3: Set Queue Units Based on Queue Name

In [6]:
import pandas as pd
import os

# ==============================
# CONFIG
# ==============================
FILE_PATH = r"C:\Users\Ranja\OneDrive\Documents\WFO_JOB_PROJECT\data\wfo_data.csv"

# ==============================
# STEP 1: LOAD DATA
# ==============================
df = pd.read_csv(FILE_PATH)

print("File loaded successfully")
print("Total rows:", len(df))
print("Total columns:", len(df.columns))

initial_row_count = len(df)

# ==============================
# STEP 2: VALIDATE REQUIRED COLUMNS
# ==============================
required_cols = ['Queue_Name', 'Queue_Units_Filter']

missing_cols = [col for col in required_cols if col not in df.columns]

if len(missing_cols) > 0:
    raise Exception(f"Missing columns: {missing_cols}")

print("Column validation passed")

# ==============================
# STEP 3: CAPTURE OLD VALUES
# ==============================
old_values = df['Queue_Units_Filter'].copy()

# ==============================
# STEP 4: STANDARDIZE TEXT
# ==============================
df['Queue_Name'] = df['Queue_Name'].astype(str).str.upper().str.strip()

# ==============================
# STEP 5: FINAL BUSINESS MAPPING
# ==============================
def map_queue_unit(queue_name):

    if 'CUSTOMER_SUPPORT' in queue_name:
        return 'Customer Support'

    if 'TECH_SUPPORT' in queue_name or 'TECH' in queue_name:
        return 'Tech Support'

    if 'INBOUND' in queue_name:
        return 'Inbound'

    if 'OUTBOUND' in queue_name:
        return 'Outbound'

    if 'RETENTION' in queue_name:
        return 'Retention'

    if 'SALES' in queue_name:
        return 'Sales'

    if 'SUPPORT' in queue_name:
        return 'Support'

    # fallback (no "Other")
    return 'Support'

df['Queue_Units_Filter'] = df['Queue_Name'].apply(map_queue_unit)

print("Queue_Units_Filter mapping applied")

# ==============================
# STEP 6: VALIDATION
# ==============================
final_row_count = len(df)

print("Initial row count:", initial_row_count)
print("Final row count:", final_row_count)

if initial_row_count != final_row_count:
    raise Exception("Row count mismatch")

changed_rows = (old_values != df['Queue_Units_Filter']).sum()

print("Rows updated:", changed_rows)

print("\nFinal distribution:")
print(df['Queue_Units_Filter'].value_counts())

# ensure no nulls
null_count = df['Queue_Units_Filter'].isna().sum()
print("\nNull values:", null_count)

if null_count > 0:
    raise Exception("Null values found after mapping")

# ==============================
# STEP 7: SAVE FILE
# ==============================
df.to_csv(FILE_PATH, index=False)

print("\nFile updated successfully at:")
print(FILE_PATH)

File loaded successfully
Total rows: 10000
Total columns: 30
Column validation passed
Queue_Units_Filter mapping applied
Initial row count: 10000
Final row count: 10000
Rows updated: 7979

Final distribution:
Support             5057
Retention           1019
Tech Support         993
Outbound             986
Inbound              974
Customer Support     971
Name: Queue_Units_Filter, dtype: int64

Null values: 0

File updated successfully at:
C:\Users\Ranja\OneDrive\Documents\WFO_JOB_PROJECT\data\wfo_data.csv


## Step 4: Fix Queue Name Prefix Using Region

In [2]:
import pandas as pd

FILE_PATH = r"C:\Users\Ranja\OneDrive\Documents\WFO_JOB_PROJECT\data\wfo_data.csv"

df = pd.read_csv(FILE_PATH)

print("File loaded successfully")

# Backup
old_values = df['Queue_Name'].copy()

# Normalize
df['Queue_Name'] = df['Queue_Name'].astype(str).str.upper().str.strip()

# ==============================
# FIX GEO PREFIX USING REGION
# ==============================
geo_prefixes = ['APAC', 'EMEA', 'AMER', 'EU', 'US', 'IN']

def fix_prefix(row):
    name = row['Queue_Name']
    region = str(row['Region']).upper()

    parts = name.split()   # split by space

    if len(parts) == 0:
        return name

    if parts[0] in geo_prefixes:
        parts[0] = region

    return " ".join(parts)

df['Queue_Name'] = df.apply(fix_prefix, axis=1)

# ==============================
# CONVERT BACK TO UNDERSCORE FORMAT
# ==============================
df['Queue_Name'] = df['Queue_Name'].str.replace(' ', '_')

print("Queue_Name corrected + underscore format restored")

# ==============================
# VALIDATION
# ==============================
changed = (old_values != df['Queue_Name']).sum()

print("Rows updated:", changed)

print("\nSample check:")
print(df[['Country', 'Region', 'Queue_Name']].head(10))

# ==============================
# SAVE
# ==============================
df.to_csv(FILE_PATH, index=False)

print("\nFile updated successfully")

File loaded successfully
Queue_Name corrected + underscore format restored
Rows updated: 10000

Sample check:
       Country Region             Queue_Name
0     Bulgaria   EMEA    INBOUND_VOICE_TIER1
1     Malaysia   APAC  TECH_SUPPORT_EMAIL_T3
2        Kenya   EMEA   EMEA_CHAT_SUPPORT_T1
3  New Zealand   APAC  APAC_EMAIL_SUPPORT_T3
4   Bangladesh   APAC  APAC_CORE_EMAIL_TIER2
5  South Korea   APAC     RETENTION_VOICE_T2
6  North Korea   APAC  APAC_CORE_EMAIL_TIER2
7      Vietnam   APAC      OUTBOUND_SALES_T2
8  New Zealand   APAC     APAC_CORE_EMAIL_T2
9  New Zealand   APAC  APAC_CORE_EMAIL_TIER2

File updated successfully
